# LM-Steer — steering replication

Replicates the steering demo from **"Word Embeddings Are Steers for Language Models"** ([arXiv:2305.12798](https://arxiv.org/abs/2305.12798)) on GPT-2.

LM-Steer learns low-rank projector pairs that perturb the final-layer hidden states feeding the LM head — `h' = h + ε · v · (h P₁)P₂ᵀ` — for **every generated token**. The checkpoint `gpt2.pt` comes from the [official demo](https://huggingface.co/spaces/Glaciohound/LM-Steer/blob/main/checkpoints/gpt2.pt) and holds four steer dimensions; dimension 2 is the sentiment axis (trained on SST-5), steered here with the demo's ε = 1e-3 and steer values ±2.

The official demo samples with `top_p=0.9` and a fixed seed; greedy decoding degenerates on GPT-2, so this notebook does the same.

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
import easysteer.vectors as vec
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = "/home/xhl/huggingface_models/openai-community/gpt2"  # openai-community/gpt2

# Declare the LM-Steer workload and the engine derives the graph
# integration (this checkpoint's rank-1000 projectors are beyond the
# in-graph rank cap, so it runs in the split tier).
llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    steer_algorithms=["lm_steer"],
)

EPSILON = 1e-3  # the checkpoint's epsilon; scale = EPSILON * steer value
payload = vec.from_lm_steer("gpt2.pt", vector_index=2)  # sentiment axis
example = "My life"
params = SamplingParams(temperature=1.0, top_p=0.9, seed=0, max_tokens=40)

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


(EngineCore pid=3764241) 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=3764241) 

...

Capturing CUDA graphs (PIECEWISE):  78%|███████▊  | 40/51 [00:01<00:00, 30.73it/s]

Capturing CUDA graphs (PIECEWISE):  86%|████████▋ | 44/51 [00:01<00:00, 28.18it/s]

Capturing CUDA graphs (PIECEWISE):  94%|█████████▍| 48/51 [00:01<00:00, 28.99it/s]

Capturing CUDA graphs (PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 28.04it/s]

Capturing CUDA graphs (PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 29.94it/s]

In [2]:
baseline = llm.generate(example, params, use_tqdm=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

=====Baseline=====
 is meant to go back to the beginning," Peeples said. "That being said, I do think it's impossible to put these things out to the public at large, because there's so


In [3]:
# The steer acts on every token's final hidden state, exactly as the
# official implementation applies it before the LM head.
for value in (2.0, -2.0):
    steering = SteeringSpec(vectors=[
        VectorSpec(
            data=payload,
            algorithm="lm_steer",
            scale=EPSILON * value,
            layers=[11],
            apply=ApplySpec(prompt="all", generation="all"),
        ),
    ])
    steered = llm.generate(example, params, steering=steering, use_tqdm=False)
    print(f"=====Sentiment steer {value:+.0f}=====")
    print(steered[0].outputs[0].text)

=====Sentiment steer +2=====
 beautifully combines the warmth and fluidity of warmth with fascinating spicy sensation. Soulful spicy deliciousness, wonderfully touching spicy delicacies and deliciously mesmerizing fascinating fascinating delicious wonderful marvelous wonderful magnificent wonderful wonderful marvelous


=====Sentiment steer -2=====
 didn't change; neither did my father's. Instead everything changed. Like everything, the bad stuff I did get worse."

This extension too can be tedious slog. Having 10 Minutes decided on
